# Notebook 7: Final End-to-End Prediction Pipeline Testing
This notebook tests our single prediction pipeline end-to-end, loading models,
classifying random MRI files, and plotting probability lists.


In [ ]:
import os
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append('..')
from ml.predict_helper import predict_brain_tumor

TEST_DIR = Path('../dataset/processed/test')
MODELS_DIR = Path('../models')
OUTPUTS_DIR = Path('temp_outputs')


### Step 1: Select a Random MRI from the Test Dataset


In [ ]:
classes = ['glioma', 'meningioma', 'notumor', 'pituitary']
selected_class = random.choice(classes)
class_dir = TEST_DIR / selected_class
image_path = random.choice(list(class_dir.glob('*')))
print(f'True Label: {selected_class.upper()} | File: {image_path.name}')


### Step 2: Run End-to-End Prediction and Grad-CAM


In [ ]:
result = predict_brain_tumor(
    image_path=str(image_path),
    model_name='Best_Model',
    models_dir=MODELS_DIR,
    outputs_dir=OUTPUTS_DIR
)

print('\nPrediction Details:')
print(f'  Model Used      : {result["model_used"]}')
print(f'  Predicted Class : {result["predicted_class"]}')
print(f'  Confidence      : {result["confidence"] * 100:.2f}%')
print(f'  Processing Time : {result["processing_time"]:.4f} seconds')
print(f'  Probabilities   :')
for cls, prob in result['probabilities'].items():
    print(f'    - {cls:<15}: {prob * 100:.2f}%')


### Step 3: Visualize Results Side-by-Side


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Original
axes[0].imshow(Image.open(image_path), cmap='gray')
axes[0].set_title(f'Original Scan (True: {selected_class.upper()})', fontweight='bold')
axes[0].axis('off')

# Grad-CAM
gradcam_img_path = OUTPUTS_DIR / f'gradcam_{image_path.name}'
axes[1].imshow(Image.open(gradcam_img_path))
axes[1].set_title(f'Predicted: {result["predicted_class"]} ({result["confidence"] * 100:.1f}%)', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Cleanup
import shutil
if OUTPUTS_DIR.exists():
    shutil.rmtree(OUTPUTS_DIR)
